# 10 — Reprodução do PortParser (baseline)

Reprodução do PortParser (Lopes et al., 2024), modelo de referência para o parsing de dependências do português brasileiro, no mesmo split do Porttinari usado nesta dissertação — a linha de base contra a qual os modelos ParseH2IA são comparados (UAS 96,08 / LAS 94,61 / UPOS 99,09).

**Pré-requisitos**: dependências em `../requirements.txt`; corpus Porttinari processado em `PARSEH2IA_DATA/data_dois/complaints_dataset_obj_outxpos` (HuggingFace `datasets`, salvo com `save_to_disk`). Por padrão os caminhos relativos `../../` assumem que este repositório está clonado dentro do diretório de dados (checkpoints e dataset no diretório pai do repositório) — ajuste a célula de configuração se necessário.

*Repositório da dissertação de mestrado — reimplementação do PortParser (ParseH2IA) para o português brasileiro, corpus Porttinari.*

# Reprodução da Arquitetura PortParser — Multi-Task Learning

Reprodução da arquitetura proposta em:
> **Towards Portparser – a highly accurate parsing system for Brazilian Portuguese following the Universal Dependencies framework**  
> Lucelene Lopes and Thiago Alexandre Salgueiro Pardo (PROPOR 2024)

**Tarefas preditas simultaneamente (multi-task learning):**

| Tarefa | Campo UD | Disponível no CSV | Implementado |
|--------|----------|-------------------|--------------|
| HEAD   | Cabeça da dependência | ✓ `head_tags`   | ✓ biaffine arc |
| DEPREL | Rótulo da relação     | ✓ `deprel_tags` | ✓ biaffine rel |
| UPOS   | Part-of-Speech        | ✓ `upos_tags`   | ✓ MLP linear   |
| UFeats | Features morfológicas | ✗ não no CSV    | — |
| Lemma  | Lema do token         | ✗ não no CSV    | — |

UFeats e Lemma não estão no dataset pré-processado. As três primeiras tarefas cobrem
o núcleo do PortParser (e são as que contribuem diretamente para UAS/LAS).

**Diferenças em relação aos notebooks de ablação:**
1. **BERTimbau Large congelado** — embeddings fixos, sem fine-tuning.
2. **BiLSTM bidirecional** (2 camadas, 512 dim cada direção) sobre a saída do BERT.
3. **Multi-task**: prediz HEAD + DEPREL + UPOS conjuntamente num único modelo.
4. **Schedule de LR em duas fases**: 1e-3 (40 épocas) → 1e-4 (20 épocas).
5. **Optimizer**: Adam puro, batch size 32.

**Nota:** O PortParser original usa UDPipe 2 (TensorFlow 1.x + Python ≤ 3.7).
Esta reimplementação usa PyTorch com os mesmos hiperparâmetros do artigo.

In [ ]:
import ast
import csv
import json
import os
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ── Constantes ───────────────────────────────────────────────────────────────
PRETRAINED_MODEL = "neuralmind/bert-large-portuguese-cased"  # BERTimbau Large

# Hiperparâmetros do artigo PortParser (Seções 3.2 e 3.4)
LSTM_DIM       = 512    # por direção → saída bidirecional: 1024
LSTM_LAYERS    = 2
LSTM_DROPOUT   = 0.33
ARC_HIDDEN     = 500
REL_HIDDEN     = 100
UPOS_HIDDEN    = 100    # MLP para UPOS (igual ao REL_HIDDEN, padrão UDPipe 2)
MLP_DROPOUT    = 0.33
BATCH_SIZE     = 32
LR_INITIAL     = 1e-3   # 40 primeiras épocas
LR_FINAL       = 1e-4   # 20 épocas finais
EPOCHS_INITIAL = 40
EPOCHS_FINAL   = 20
TOTAL_EPOCHS   = EPOCHS_INITIAL + EPOCHS_FINAL
MAX_LENGTH     = 512
GRAD_CLIP      = 1.0

DEPREL_LABELS = [
    'det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case',
    'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop',
    'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj',
    'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer',
    'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated',
    'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list',
    'reparandum', 'csubj:pass',
]
NUM_DEPREL = len(DEPREL_LABELS)

# Mapeamento UPOS derivado do dataset (índice → string)
UPOS_LABELS = [
    'DET', 'NOUN', 'VERB', 'PUNCT', 'SCONJ', 'ADP',
    'ADJ', 'CCONJ', 'ADV', 'PROPN', 'AUX', 'NUM',
    'PRON', 'SYM', 'X', 'INTJ',
]
NUM_UPOS = len(UPOS_LABELS)

print(f"DEPREL labels: {NUM_DEPREL} | UPOS labels: {NUM_UPOS}")

In [ ]:
# ── Carregamento de dados ─────────────────────────────────────────────────────
DATA_DIR = "../../data_dois"

def load_csv(path):
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            rows.append({
                "tokens":      ast.literal_eval(row["tokens"]),
                "head_tags":   ast.literal_eval(row["head_tags"]),
                "deprel_tags": ast.literal_eval(row["deprel_tags"]),
                "upos_tags":   ast.literal_eval(row["upos_tags"]),
            })
    return rows

train_data = load_csv(os.path.join(DATA_DIR, "train_outxpos.csv"))
val_data   = load_csv(os.path.join(DATA_DIR, "val_outxpos.csv"))
test_data  = load_csv(os.path.join(DATA_DIR, "test_outxpos.csv"))
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

In [ ]:
# ── Dataset e Collate ────────────────────────────────────────────────────────
class PortParserDataset(Dataset):
    """Representação no nível de PALAVRA via primeira subpalavra BERT."""

    def __init__(self, data, tokenizer, max_length=512):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item   = self.data[idx]
        tokens = item["tokens"]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors=None,
        )
        word_ids = encoding.word_ids()

        first_subword: dict[int, int] = {}
        for tok_pos, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx not in first_subword:
                first_subword[word_idx] = tok_pos

        valid_words = sorted(first_subword.keys())

        return {
            "input_ids":     encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "first_subword": [first_subword[w]         for w in valid_words],
            "head_labels":   [item["head_tags"][w]     for w in valid_words],
            "deprel_labels": [item["deprel_tags"][w]   for w in valid_words],
            "upos_labels":   [item["upos_tags"][w]     for w in valid_words],
            "num_words":     len(valid_words),
        }


def collate_fn(batch):
    max_tokens = max(len(b["input_ids"])  for b in batch)
    max_words  = max(b["num_words"]       for b in batch)

    input_ids      = torch.zeros(len(batch), max_tokens, dtype=torch.long)
    attention_mask = torch.zeros(len(batch), max_tokens, dtype=torch.long)
    first_subword  = torch.zeros(len(batch), max_words,  dtype=torch.long)
    head_labels    = torch.full((len(batch), max_words), -100, dtype=torch.long)
    deprel_labels  = torch.full((len(batch), max_words), -100, dtype=torch.long)
    upos_labels    = torch.full((len(batch), max_words), -100, dtype=torch.long)
    word_mask      = torch.zeros(len(batch), max_words,  dtype=torch.bool)

    for i, b in enumerate(batch):
        T = len(b["input_ids"])
        W = b["num_words"]
        input_ids[i, :T]      = torch.tensor(b["input_ids"],      dtype=torch.long)
        attention_mask[i, :T] = torch.tensor(b["attention_mask"], dtype=torch.long)
        first_subword[i, :W]  = torch.tensor(b["first_subword"],  dtype=torch.long)
        head_labels[i, :W]    = torch.tensor(b["head_labels"],    dtype=torch.long)
        deprel_labels[i, :W]  = torch.tensor(b["deprel_labels"],  dtype=torch.long)
        upos_labels[i, :W]    = torch.tensor(b["upos_labels"],    dtype=torch.long)
        word_mask[i, :W]      = True

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "first_subword":  first_subword,
        "head_labels":    head_labels,
        "deprel_labels":  deprel_labels,
        "upos_labels":    upos_labels,
        "word_mask":      word_mask,
    }

In [ ]:
# ── Blocos de arquitetura (biaffine) ─────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, in_features: int, out_features: int, dropout: float = 0.33):
        super().__init__()
        self.linear     = nn.Linear(in_features, out_features)
        self.activation = nn.ELU()
        self.norm       = nn.LayerNorm(out_features)
        self.dropout    = nn.Dropout(dropout)
        nn.init.orthogonal_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.norm(self.activation(self.linear(x))))


class Biaffine(nn.Module):
    def __init__(self, in_features: int, out_features: int = 1,
                 bias_x: bool = True, bias_y: bool = True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.bias_x = bias_x
        self.bias_y = bias_y
        self.weight = nn.Parameter(torch.zeros(
            out_features,
            in_features + int(bias_x),
            in_features + int(bias_y),
        ))
        nn.init.normal_(self.weight, std=1.0 / in_features)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)
        # x: [B, W, H+1]  W: [out, H+1, H+1]  → [B, out, W, W]
        return torch.einsum('bih,ohk,bjk->boij', x, self.weight, y)

In [ ]:
# ── Modelo PortParser (Multi-Task) ───────────────────────────────────────────
class PortParserModel(nn.Module):
    """Reimplementação fiel da arquitetura PortParser/UDPipe 2 em PyTorch.

    Pipeline:
      1. BERTimbau Large (CONGELADO) → embeddings de subpalavras
      2. Extração da primeira subpalavra → representação por PALAVRA
      3. BiLSTM bidirecional (2 camadas, 512 dim) → representação contextualizada
      4. Três cabeças de predição (multi-task):
         - Biaffine arc  → HEAD
         - Biaffine rel  → DEPREL
         - MLP linear    → UPOS
    """

    def __init__(
        self,
        bert_model_name:   str,
        num_deprel_labels: int,
        num_upos_labels:   int,
        lstm_dim:     int   = 512,
        lstm_layers:  int   = 2,
        lstm_dropout: float = 0.33,
        arc_hidden:   int   = 500,
        rel_hidden:   int   = 100,
        upos_hidden:  int   = 100,
        mlp_dropout:  float = 0.33,
    ):
        super().__init__()

        # 1. Encoder BERT — congelado (embeddings pré-computados)
        self.bert = AutoModel.from_pretrained(bert_model_name)
        for param in self.bert.parameters():
            param.requires_grad = False
        bert_dim = self.bert.config.hidden_size  # 1024 para BERTimbau Large

        # 2. BiLSTM — único componente treinado sobre as features do BERT
        self.lstm = nn.LSTM(
            input_size=bert_dim,
            hidden_size=lstm_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=lstm_dropout if lstm_layers > 1 else 0.0,
        )
        D = lstm_dim * 2  # 1024 (bidirecional)

        # 3a. HEAD — biaffine arc
        self.arc_dep_mlp  = MLP(D, arc_hidden, mlp_dropout)
        self.arc_head_mlp = MLP(D, arc_hidden, mlp_dropout)
        self.arc_biaffine = Biaffine(arc_hidden, out_features=1,
                                     bias_x=True, bias_y=False)

        # 3b. DEPREL — biaffine rel
        self.rel_dep_mlp  = MLP(D, rel_hidden, mlp_dropout)
        self.rel_head_mlp = MLP(D, rel_hidden, mlp_dropout)
        self.rel_biaffine = Biaffine(rel_hidden, out_features=num_deprel_labels,
                                     bias_x=True, bias_y=True)

        # 3c. UPOS — MLP linear (tarefa mais simples, não precisa de biaffine)
        self.upos_mlp        = MLP(D, upos_hidden, mlp_dropout)
        self.upos_classifier = nn.Linear(upos_hidden, num_upos_labels)

        self.dropout       = nn.Dropout(mlp_dropout)
        self.num_deprel    = num_deprel_labels
        self.num_upos      = num_upos_labels

    def _word_reprs(self, bert_out, first_subword):
        """[B, T, D] × [B, W] → [B, W, D]: primeira subpalavra de cada palavra."""
        B, W = first_subword.shape
        idx  = first_subword.unsqueeze(-1).expand(B, W, bert_out.size(-1))
        return bert_out.gather(1, idx)

    def forward(self, input_ids, attention_mask, first_subword, word_mask,
                head_labels=None, deprel_labels=None, upos_labels=None):

        # 1. BERT (sem gradiente)
        with torch.no_grad():
            bert_out = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
            ).last_hidden_state                         # [B, T, D_bert]

        # 2. Representação por palavra → BiLSTM
        h = self.dropout(self.lstm(self._word_reprs(bert_out, first_subword))[0])
        B, W, _ = h.shape                               # [B, W, 1024]

        # 3a. HEAD
        logits_head = self.arc_biaffine(
            self.arc_dep_mlp(h), self.arc_head_mlp(h)
        ).squeeze(1)                                    # [B, W, W]
        logits_head = logits_head.masked_fill((~word_mask).unsqueeze(1), -1e4)

        # 3b. DEPREL
        # Teacher forcing no treino; predição greedy no eval
        if head_labels is not None and self.training:
            head_idx = head_labels.clone().clamp_(0, W - 1)
            head_idx[head_labels == -100] = 0
        else:
            head_idx = logits_head.argmax(-1)           # [B, W]

        # Matriz biaffine completa → indexa pela HEAD de cada palavra
        logits_rel_full = self.rel_biaffine(
            self.rel_dep_mlp(h), self.rel_head_mlp(h)
        ).permute(0, 2, 3, 1)                          # [B, W_dep, W_head, num_deprel]
        hi_exp     = head_idx.unsqueeze(-1).unsqueeze(-1).expand(B, W, 1, self.num_deprel)
        logits_rel = logits_rel_full.gather(2, hi_exp).squeeze(2)  # [B, W, num_deprel]

        # 3c. UPOS
        logits_upos = self.upos_classifier(self.upos_mlp(h))       # [B, W, num_upos]

        # Perdas (multi-task: soma igual ao UDPipe 2)
        loss = None
        if head_labels is not None and deprel_labels is not None and upos_labels is not None:
            ce = nn.CrossEntropyLoss(ignore_index=-100)

            head_lbl = head_labels.clone().clamp_(0, W - 1)
            head_lbl[~word_mask] = -100

            loss = (
                ce(logits_head.reshape(-1, W),           head_lbl.reshape(-1))
                + ce(logits_rel.reshape(-1, self.num_deprel), deprel_labels.reshape(-1))
                + ce(logits_upos.reshape(-1, self.num_upos),  upos_labels.reshape(-1))
            )

        return loss, logits_head, logits_rel, logits_upos

In [ ]:
# ── Avaliação (UAS / LAS / UPOS-acc) ─────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss = tot_arc = tot_lab = tot_upos = tot_n = 0

    for batch in loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        loss, logits_head, logits_rel, logits_upos = model(
            input_ids      = batch["input_ids"],
            attention_mask = batch["attention_mask"],
            first_subword  = batch["first_subword"],
            word_mask      = batch["word_mask"],
            head_labels    = batch["head_labels"],
            deprel_labels  = batch["deprel_labels"],
            upos_labels    = batch["upos_labels"],
        )

        valid = batch["word_mask"] & (batch["head_labels"] != -100)
        n     = valid.sum().item()
        W     = logits_head.size(-1)

        head_lbl     = batch["head_labels"].clamp(0, W - 1)
        head_preds   = logits_head.argmax(-1)
        deprel_preds = logits_rel.argmax(-1)
        upos_preds   = logits_upos.argmax(-1)

        tot_arc  += ((head_preds == head_lbl) & valid).sum().item()
        tot_lab  += ((head_preds == head_lbl) & (deprel_preds == batch["deprel_labels"]) & valid).sum().item()
        tot_upos += ((upos_preds == batch["upos_labels"]) & valid).sum().item()
        tot_n    += n
        if loss is not None:
            tot_loss += loss.item() * n

    uas      = tot_arc  / tot_n if tot_n else 0.0
    las      = tot_lab  / tot_n if tot_n else 0.0
    upos_acc = tot_upos / tot_n if tot_n else 0.0
    avg_loss = tot_loss / tot_n if tot_n else 0.0
    return avg_loss, uas, las, upos_acc

In [ ]:
# ── Treinamento ───────────────────────────────────────────────────────────────
def train_portparser():
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)

    train_ds = PortParserDataset(train_data, tokenizer, MAX_LENGTH)
    val_ds   = PortParserDataset(val_data,   tokenizer, MAX_LENGTH)
    test_ds  = PortParserDataset(test_data,  tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,
                              collate_fn=collate_fn, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                              collate_fn=collate_fn, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                              collate_fn=collate_fn, num_workers=4, pin_memory=True)

    model = PortParserModel(
        bert_model_name   = PRETRAINED_MODEL,
        num_deprel_labels = NUM_DEPREL,
        num_upos_labels   = NUM_UPOS,
        lstm_dim          = LSTM_DIM,
        lstm_layers       = LSTM_LAYERS,
        lstm_dropout      = LSTM_DROPOUT,
        arc_hidden        = ARC_HIDDEN,
        rel_hidden        = REL_HIDDEN,
        upos_hidden       = UPOS_HIDDEN,
        mlp_dropout       = MLP_DROPOUT,
    ).to(device)

    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"Parâmetros treináveis : {sum(p.numel() for p in trainable):,}")
    print(f"Parâmetros BERT (fixos): {sum(p.numel() for p in model.bert.parameters()):,}")

    optimizer = Adam(trainable, lr=LR_INITIAL)

    best_las  = -1.0
    best_path = "../../best_models_portparser"
    os.makedirs(best_path, exist_ok=True)
    log_rows  = []

    for epoch in range(1, TOTAL_EPOCHS + 1):
        # Duas fases de LR (artigo: 40-20 épocas)
        if epoch == EPOCHS_INITIAL + 1:
            for g in optimizer.param_groups:
                g["lr"] = LR_FINAL
            print(f"  [Época {epoch}] LR reduzido para {LR_FINAL}")

        # ── Treino ──────────────────────────────────────────────────────────
        model.train()
        tot_loss = tot_n = 0

        for batch in train_loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
            optimizer.zero_grad()
            loss, *_ = model(
                input_ids      = batch["input_ids"],
                attention_mask = batch["attention_mask"],
                first_subword  = batch["first_subword"],
                word_mask      = batch["word_mask"],
                head_labels    = batch["head_labels"],
                deprel_labels  = batch["deprel_labels"],
                upos_labels    = batch["upos_labels"],
            )
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            optimizer.step()

            n         = batch["word_mask"].sum().item()
            tot_loss += loss.item() * n
            tot_n    += n

        train_loss = tot_loss / tot_n if tot_n else 0.0

        # ── Validação ────────────────────────────────────────────────────────
        val_loss, val_uas, val_las, val_upos = evaluate(model, val_loader)

        log_rows.append({
            "Epoch":           epoch,
            "Training Loss":   train_loss,
            "Validation Loss": val_loss,
            "Uas":             val_uas,
            "Las":             val_las,
            "Upos":            val_upos,
        })

        print(f"Época {epoch:3d}/{TOTAL_EPOCHS} | "
              f"Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"UAS: {val_uas:.4f} | LAS: {val_las:.4f} | UPOS: {val_upos:.4f}")

        if val_las > best_las:
            best_las = val_las
            torch.save(model.state_dict(), os.path.join(best_path, "best_model.pt"))
            print(f"  → Melhor modelo salvo (LAS: {best_las:.4f})")

    # ── Log CSV ────────────────────────────────────────────────────────────────
    log_csv = "../../training_log_portparser_neuralmind_bert-large-portuguese-cased.csv"
    pd.DataFrame(log_rows).to_csv(log_csv, index=False)
    print(f"\nLog salvo em: {log_csv}")

    # ── Avaliação final no teste ───────────────────────────────────────────────
    model.load_state_dict(torch.load(os.path.join(best_path, "best_model.pt"), map_location=device))
    test_loss, test_uas, test_las, test_upos = evaluate(model, test_loader)
    print(f"\n{'='*65}")
    print(f"  Avaliação final no TEST SET (melhor modelo)")
    print(f"  UAS: {test_uas:.6f} | LAS: {test_las:.6f} | UPOS: {test_upos:.6f}")
    print(f"{'='*65}")

    result_dict = {
        "name":           PRETRAINED_MODEL,
        "architecture":   "portparser_bilstm_biaffine_frozen_bert_multitask",
        "hyperparameters": {
            "lstm_dim":       LSTM_DIM,
            "lstm_layers":    LSTM_LAYERS,
            "arc_hidden":     ARC_HIDDEN,
            "rel_hidden":     REL_HIDDEN,
            "upos_hidden":    UPOS_HIDDEN,
            "lr_initial":     LR_INITIAL,
            "lr_final":       LR_FINAL,
            "epochs_initial": EPOCHS_INITIAL,
            "epochs_final":   EPOCHS_FINAL,
            "batch_size":     BATCH_SIZE,
        },
        "uas":      test_uas,
        "las":      test_las,
        "upos_acc": test_upos,
    }

    results_file = "../../portparser_results.jsonl"
    with open(results_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")
    print(f"Resultado salvo em: {results_file}")

    return result_dict

In [ ]:
# ── Execução ─────────────────────────────────────────────────────────────────
result = train_portparser()
print(f"\nResultado final: UAS={result['uas']:.4f} | LAS={result['las']:.4f} | UPOS={result['upos_acc']:.4f}")

In [ ]:
# ── Formatação LaTeX (mesmo padrão do format_training_logs.py) ───────────────
log_csv = "../../training_log_portparser_neuralmind_bert-large-portuguese-cased.csv"
log_df  = pd.read_csv(log_csv)

print("% portparser_neuralmind_bert-large-portuguese-cased  (multi-task: HEAD + DEPREL + UPOS)")
for _, row in log_df.iterrows():
    epoch      = int(row["Epoch"])
    train_loss = float(row["Training Loss"])
    val_loss   = float(row["Validation Loss"])
    uas        = float(row["Uas"])
    las        = float(row["Las"])
    upos       = float(row["Upos"])
    print(f"{epoch} & {train_loss:.6f} & {val_loss:.6f} & {uas:.6f} & {las:.6f} & {upos:.6f} \\\\")

# Linha do evaluate() final (melhor modelo no teste)
results_file = "../../portparser_results.jsonl"
with open(results_file) as f:
    ev = json.loads(f.readlines()[-1])
print(f"% evaluate()  UAS={ev['uas']:.6f}  LAS={ev['las']:.6f}  UPOS={ev['upos_acc']:.6f}")
print(r"\hline")
print(f"best & -- & -- & {ev['uas']:.6f} & {ev['las']:.6f} & {ev['upos_acc']:.6f} \\\\")

In [ ]:
# ── Inferência no conjunto de teste ──────────────────────────────────────────
# Célula independente: carrega o melhor modelo salvo e avalia no teste,
# gerando métricas globais + breakdown por classe de DEPREL e UPOS.

import collections

BEST_MODEL_PATH = "../../best_models_portparser/best_model.pt"

# ── 1. Reconstrói e carrega o modelo ─────────────────────────────────────────
tokenizer_inf = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)

model_inf = PortParserModel(
    bert_model_name   = PRETRAINED_MODEL,
    num_deprel_labels = NUM_DEPREL,
    num_upos_labels   = NUM_UPOS,
    lstm_dim          = LSTM_DIM,
    lstm_layers       = LSTM_LAYERS,
    lstm_dropout      = LSTM_DROPOUT,
    arc_hidden        = ARC_HIDDEN,
    rel_hidden        = REL_HIDDEN,
    upos_hidden       = UPOS_HIDDEN,
    mlp_dropout       = MLP_DROPOUT,
).to(device)

model_inf.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model_inf.eval()
print(f"Modelo carregado de: {BEST_MODEL_PATH}")

# ── 2. DataLoader de teste ────────────────────────────────────────────────────
test_ds_inf     = PortParserDataset(test_data, tokenizer_inf, MAX_LENGTH)
test_loader_inf = DataLoader(test_ds_inf, batch_size=64, shuffle=False,
                             collate_fn=collate_fn, num_workers=4, pin_memory=True)

# ── 3. Coleta predições palavra a palavra ─────────────────────────────────────
all_head_gold,   all_head_pred   = [], []
all_deprel_gold, all_deprel_pred = [], []
all_upos_gold,   all_upos_pred   = [], []

with torch.no_grad():
    for batch in test_loader_inf:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                 for k, v in batch.items()}

        _, logits_head, logits_rel, logits_upos = model_inf(
            input_ids      = batch["input_ids"],
            attention_mask = batch["attention_mask"],
            first_subword  = batch["first_subword"],
            word_mask      = batch["word_mask"],
        )

        valid = batch["word_mask"] & (batch["head_labels"] != -100)
        W     = logits_head.size(-1)
        flat  = valid.reshape(-1)

        all_head_gold.extend(   batch["head_labels"].clamp(0, W-1).reshape(-1)[flat].cpu().tolist())
        all_head_pred.extend(   logits_head.argmax(-1).reshape(-1)[flat].cpu().tolist())
        all_deprel_gold.extend( batch["deprel_labels"].reshape(-1)[flat].cpu().tolist())
        all_deprel_pred.extend( logits_rel.argmax(-1).reshape(-1)[flat].cpu().tolist())
        all_upos_gold.extend(   batch["upos_labels"].reshape(-1)[flat].cpu().tolist())
        all_upos_pred.extend(   logits_upos.argmax(-1).reshape(-1)[flat].cpu().tolist())

N = len(all_head_gold)
print(f"Tokens avaliados: {N:,}")

# ── 4. Métricas globais ───────────────────────────────────────────────────────
arc_ok  = sum(g == p for g, p in zip(all_head_gold, all_head_pred))
lab_ok  = sum(
    (g_h == p_h) and (g_d == p_d)
    for g_h, p_h, g_d, p_d
    in zip(all_head_gold, all_head_pred, all_deprel_gold, all_deprel_pred)
)
upos_ok = sum(g == p for g, p in zip(all_upos_gold, all_upos_pred))

uas_test  = arc_ok  / N
las_test  = lab_ok  / N
upos_test = upos_ok / N

print(f"\n{'='*55}")
print(f"  MÉTRICAS GLOBAIS — TEST SET")
print(f"  UAS  : {uas_test:.4f}  ({arc_ok:,}/{N:,})")
print(f"  LAS  : {las_test:.4f}  ({lab_ok:,}/{N:,})")
print(f"  UPOS : {upos_test:.4f}  ({upos_ok:,}/{N:,})")
print(f"{'='*55}")

# ── 5. Acurácia por classe DEPREL ─────────────────────────────────────────────
deprel_totals  = collections.Counter(all_deprel_gold)
deprel_correct = collections.Counter(
    g for g, p in zip(all_deprel_gold, all_deprel_pred) if g == p
)

deprel_rows = [
    {
        "Classe":  DEPREL_LABELS[idx],
        "Total":   deprel_totals[idx],
        "Correto": deprel_correct.get(idx, 0),
        "Acc":     deprel_correct.get(idx, 0) / deprel_totals[idx],
    }
    for idx in sorted(deprel_totals)
]
deprel_df = pd.DataFrame(deprel_rows).sort_values("Total", ascending=False)

print("\n── Acurácia por DEPREL (top 20 por frequência) ─────────────")
print(f"{'Classe':<22} {'Total':>7} {'Correto':>8} {'Acc':>7}")
print("-" * 47)
for _, r in deprel_df.head(20).iterrows():
    print(f"{r['Classe']:<22} {r['Total']:>7,} {r['Correto']:>8,} {r['Acc']:>7.4f}")

# ── 6. Acurácia por classe UPOS ───────────────────────────────────────────────
upos_totals  = collections.Counter(all_upos_gold)
upos_correct = collections.Counter(
    g for g, p in zip(all_upos_gold, all_upos_pred) if g == p
)

upos_rows = [
    {
        "Classe":  UPOS_LABELS[idx],
        "Total":   upos_totals[idx],
        "Correto": upos_correct.get(idx, 0),
        "Acc":     upos_correct.get(idx, 0) / upos_totals[idx],
    }
    for idx in sorted(upos_totals)
]
upos_df = pd.DataFrame(upos_rows).sort_values("Total", ascending=False)

print("\n── Acurácia por UPOS ────────────────────────────────────────")
print(f"{'Classe':<10} {'Total':>7} {'Correto':>8} {'Acc':>7}")
print("-" * 37)
for _, r in upos_df.iterrows():
    print(f"{r['Classe']:<10} {r['Total']:>7,} {r['Correto']:>8,} {r['Acc']:>7.4f}")